# Correlation matrices (structures + cross-assets)

1. **Level corr** — pairwise correlation on `CORR_SERIES` (últimas `CORR_WINDOW` barras)
2. **Corr percentile** — cuán extremo es el corr actual vs los últimos `CORR_Z_LOOKBACK` rolling corrs → heatmap

Edita en `curve_config.py`:
- `CROSS_ASSET` — tickers BBG extra (`SPX`, `CO1`, …)
- `CORR_SERIES` — columnas a incluir en las matrices
- `CORR_WINDOW`, `CORR_PCT_LOOKBACK`

Tras añadir cross-assets: `git pull` + `--pull-history` para traerlos al live store.

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "correlations.py").exists() else _DZ
if not (REPO / "correlations.py").exists():
    raise FileNotFoundError("No encuentro correlations.py en %s — haz git pull" % REPO)

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
import curve_config
import correlations

importlib.reload(curve_config)
importlib.reload(correlations)

print("CROSS_ASSET:", curve_config.CROSS_ASSET)
print("CORR_SERIES:", curve_config.CORR_SERIES)
print("window=%s  pct_lookback=%s" % (
    curve_config.CORR_WINDOW, curve_config.CORR_PCT_LOOKBACK,
))

In [ ]:
level, zmat, full = correlations.run_correlations()
print("panel cols sample:", list(full.columns)[:20], "...")
print("asof:", full.index[-1], "n=", len(full))

In [ ]:
# ── 1) Matriz de correlaciones (nivel) ───────────────────────────────────────
display(level.round(3))

In [ ]:
# ── 2) Percentil del rolling correlation + heatmap ─────────────────────────────
display(zmat.round(2))

heat = Path(curve_config.GRAPH_DIR_CORR) / "corr_percentile_heatmap.png"
if heat.exists():
    display(Image(filename=str(heat)))
else:
    print("no heatmap — check matplotlib")

In [ ]:
# Level heatmap (opcional)
heat_l = Path(curve_config.GRAPH_DIR_CORR) / "corr_level_heatmap.png"
if heat_l.exists():
    display(Image(filename=str(heat_l)))